In [ ]:
import glob
import os

import pandas as pd
import numpy as np

import tifffile as tf
from tifffile import imread
from tifffile import imwrite

import skimage
from skimage.util import compare_images
from skimage import io, color, measure
from skimage.measure import regionprops
from skimage.measure import label as sk_label  # Renaming the label function to avoid conflicts

import matplotlib.pyplot as plt
import matplotlib

from tqdm import tqdm

%load_ext autoreload
%autoreload 2
%matplotlib inline

In [13]:
# Import images

images_dir = '/Volumes/bamfaile/Franklin/20241012/ACTB-KI_NGN2_Batch20241005_D7/Puro10min/Zstack'
images_path = os.path.join(images_dir,'*.stk') 
images_files = glob.glob(images_path)

In [ ]:
print(images_dir)
print(images_path)
print(len(images_files))

In [ ]:
# Sorts images in list alphabetically

images_files_sorted = np.sort(images_files)

print(len(images_files_sorted))
images_files_sorted

In [16]:
images_files_halo = [f for f in images_files_sorted if 'conf561' in f]
images_files_tubulin = [f for f in images_files_sorted if 'conf488' in f]
images_files_actin = [f for f in images_files_sorted if 'conf640' in f]

In [ ]:
# Initialize lists to store images by channel
images_halo = []
images_tubulin = []
images_actin = []

# Sort images into respective lists based on filename cues
for file in tqdm(images_files_sorted):
    if 'conf561' in file:
        images_halo.append(imread(file))
    elif 'conf488' in file:
        images_tubulin.append(imread(file))
    elif 'conf640' in file:
        images_actin.append(imread(file))

# Check the lists
print(f"Halo images: {len(images_halo)}")
print(f"Tubulin images: {len(images_tubulin)}")
print(f"Actin images: {len(images_actin)}")

In [ ]:
np.unique(images_actin[0])

In [ ]:
# Function to create a max projection for slices 8 to 10
def max_projection(image, start_slice=7, end_slice=11):
    return np.max(image[start_slice:end_slice+1], axis=0)

# Apply max projection to each list
max_proj_halo = [max_projection(img) for img in images_halo]
max_proj_tubulin = [max_projection(img) for img in images_tubulin]
max_proj_actin = [max_projection(img) for img in images_actin]

# Check the number of max projections created
print(f"Max projections for Halo: {len(max_proj_halo)}")
print(f"Max projections for Tubulin: {len(max_proj_tubulin)}")
print(f"Max projections for Actin: {len(max_proj_actin)}")

In [ ]:
# Plot whole cell and nucleus mask side by side

fig, ax = plt.subplots(1, 3, figsize = (12,4))
ax[0].imshow(max_proj_halo[2])
ax[1].imshow(max_proj_tubulin[2])
ax[2].imshow(max_proj_actin[2])

In [10]:
# Output directory max projections

max_project_dir = r'/Volumes/gchao/bamfaile/Analysis/ACTB-KI/00_Batch20241005_iNs/D7/Puro10min/Max_projections_seperate'


In [ ]:
# Save max projections to seperate folders

# Define the folder names
folders = ['Halo', 'Tubulin', 'Actin']

# Create the folders if they don't exist
for folder in folders:
    folder_path = os.path.join(max_project_dir, folder)
    os.makedirs(folder_path, exist_ok=True)

# Function to save a max projection image
def save_max_projection_image(image, original_file, channel_name):
    # Extract the file name (without path)
    file_name = os.path.basename(original_file)
    
    # Extract the base name (remove everything after the last '_')
    base_name = file_name.rsplit('_', 1)[0]  # This splits at the last underscore
    
    # Create the new filename with the appropriate suffix (e.g., "_halo", "_tubulin", "_actin")
    new_file_name = base_name + f"_{channel_name}.tif"
    
    # Define the output path
    output_file = os.path.join(max_project_dir, channel_name, new_file_name)
    
    # Check if file already exists to prevent overwriting
    if os.path.exists(output_file):
        print(f"Warning: {output_file} already exists! Skipping...")
        return
    
    # Save the max projection image to the corresponding folder
    tf.imwrite(output_file, image)
    print(f"Saved {output_file}")

# Now save the max projections for each channel
for i, image_file in enumerate(images_files_halo):
    save_max_projection_image(max_proj_halo[i], image_file, 'halo')

for i, image_file in enumerate(images_files_tubulin):
    save_max_projection_image(max_proj_tubulin[i], image_file, 'tubulin')

for i, image_file in enumerate(images_files_actin):
    save_max_projection_image(max_proj_actin[i], image_file, 'actin')


In [211]:
# Import masks

actin_masks_dir = '/Volumes/Analysis/ACTB-KI/00_Batch20241011_iNs/D10/Unperturbed/Max_projections_seperate/Actin/Filtered/Masks'
actin_masks_path = os.path.join(actin_masks_dir,'*.tif') 
actin_masks_files = glob.glob(actin_masks_path)

In [ ]:
print(actin_masks_dir)
print(actin_masks_path)
print(len(actin_masks_files))


In [ ]:
# Sorts images in list alphabetically

actin_masks_files_sorted = np.sort(actin_masks_files)

print(len(actin_masks_files_sorted))
actin_masks_files_sorted

### Trying to make cleaner images in the 561 channel by filtering with actin mask

Results in holes in the nuclei, try ilastik directly on whole 561 images first.

In [214]:
# Read images into list

actin_masks = []

for file in actin_masks_files_sorted:
    image = imread(file)
    actin_masks.append(image)

In [ ]:
np.unique(actin_masks[0])

In [ ]:
# Plot whole cell and nucleus mask side by side

fig, ax = plt.subplots(1, 3, figsize = (12,4))
ax[0].imshow(actin_masks[3])
ax[1].imshow(actin_masks[4])
ax[2].imshow(actin_masks[5])

In [217]:
# Import images

halo_max_dir = '/Volumes/Analysis/ACTB-KI/00_Batch20241011_iNs/D10/Unperturbed/Max_projections_seperate/Halo'
halo_max_path = os.path.join(halo_max_dir,'*.tif') 
halo_max_files = glob.glob(halo_max_path)

In [ ]:
print(halo_max_dir)
print(halo_max_path)
print(len(halo_max_files))


In [ ]:
# Sorts images in list alphabetically

halo_max_files_sorted = np.sort(halo_max_files)

print(len(halo_max_files_sorted))
halo_max_files_sorted

In [220]:
# Read images into list

max_proj_halo = []

for file in halo_max_files_sorted:
    image = imread(file)
    max_proj_halo.append(image)

In [ ]:
# Plot whole cell and nucleus mask side by side

fig, ax = plt.subplots(1, 3, figsize = (12,4))
ax[0].imshow(max_proj_halo[3])
ax[1].imshow(max_proj_halo[4])
ax[2].imshow(max_proj_halo[5])

In [222]:
import numpy as np
import tifffile as tf

# Ensure both lists have the same length, as we're pairing corresponding images
assert len(max_proj_halo) == len(actin_masks), "Mismatch between max_proj_halo and actin_masks list lengths."

# List to store the masked halo images
masked_halo_images = []

for halo_img, mask_img in zip(max_proj_halo, actin_masks):
    # Convert the mask to a binary mask (1 where mask is non-zero, 0 where mask is zero)
    binary_mask = (mask_img > 0).astype(np.uint16)  # Convert to uint16 to match halo image type
    
    # Apply the mask to the halo image (multiplying zeroed mask areas will zero the halo image there)
    masked_halo = halo_img * binary_mask
    
    # Store the masked image
    masked_halo_images.append(masked_halo)

In [ ]:
# Plot masked images

fig, ax = plt.subplots(1, 3, figsize = (12,4))
ax[0].imshow(masked_halo_images[0])
ax[1].imshow(masked_halo_images[1])
ax[2].imshow(masked_halo_images[2])

In [225]:
# Output directory max projections

masked_halo_dir = r'/Volumes/Analysis/ACTB-KI/00_Batch20241011_iNs/D10/Unperturbed/Max_projections_seperate/Halo/Masked_halo'

In [ ]:
# Save filtered images

for img, tiff_file in zip(masked_halo_images, halo_max_files_sorted):
    
    # Extract the original file name without the extension
    file_name = os.path.splitext(os.path.basename(tiff_file))[0]
    print(file_name)
    
    # Define the output file path
    output_file = os.path.join(masked_halo_dir, file_name + '_masked.tif')
    print(output_file)
    
    # Save the manipulated image as a TIFF
    tf.imwrite(output_file, img)

In [ ]:
# Optionally, save masked images if needed
output_dir = '/path/to/save/masked_halo_images'  # Replace with your desired output directory
os.makedirs(output_dir, exist_ok=True)

for i, masked_img in enumerate(masked_halo_images):
    output_path = os.path.join(output_dir, f"masked_halo_{i}.tif")
    tf.imwrite(output_path, masked_img)
    print(f"Saved masked image to {output_path}")